# Credit Risk Classification Data Preparation

To successfully run the Credit Risk Assessment classification section, we need appropriate data. Here are multiple options to obtain credit risk data:

## Option 1: Use Public Credit Risk Datasets

There are several publicly available credit risk datasets you can use:

### 1.1 Lending Club Dataset

This is a comprehensive dataset containing loan data for all loans issued by Lending Club.

```markdown
1. Visit Kaggle: https://www.kaggle.com/datasets/wordsforthewise/lending-club
2. Download the CSV file (note: this is a large dataset ~1.7GB)
3. Save as 'credit_risk_data.csv' in your working directory
```

### 1.2 Home Credit Default Risk Dataset

```markdown
1. Visit Kaggle: https://www.kaggle.com/c/home-credit-default-risk/data
2. Download the application_train.csv file
3. Rename to 'credit_risk_data.csv' and save in your working directory
```

### 1.3 German Credit Dataset (Smaller Option)

```markdown
1. Direct UCI link: https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data
2. This will need preprocessing (see code below)
```

## Option 2: Generate Synthetic Credit Risk Data

If you prefer not to download external datasets, you can generate synthetic data that matches the required structure:

```python
# Generate synthetic credit risk data
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from faker import Faker
import random
import os

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)
fake = Faker()
Faker.seed(42)

def generate_credit_risk_data(n_samples=10000, file_path='credit_risk_data.csv'):
    """
    Generate synthetic credit risk data and save as CSV
    
    Parameters:
    -----------
    n_samples: int, number of samples to generate
    file_path: str, path to save the CSV file
    
    Returns:
    --------
    DataFrame with synthetic credit risk data
    """
    print(f"Generating synthetic credit risk dataset with {n_samples} samples...")
    
    # Generate binary target with class imbalance (15% default rate)
    X, y = make_classification(
        n_samples=n_samples,
        n_features=10,
        n_informative=8,
        n_redundant=2,
        weights=[0.85, 0.15],  # 15% default rate
        random_state=42
    )
    
    # Create dataframe with features
    df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(10)])
    
    # Add customer IDs
    df['Customer ID'] = [f'CUST-{i:05d}' for i in range(n_samples)]
    
    # Add loan_default target column
    df['loan_default'] = y
    
    # Generate realistic features based on the synthetic data
    # Scale feature_0 to loan_amount (5,000 to 50,000)
    df['loan_amount'] = 5000 + (df['feature_0'] - df['feature_0'].min()) / \
                       (df['feature_0'].max() - df['feature_0'].min()) * 45000
    
    # Scale feature_1 to annual_income (25,000 to 250,000)
    df['annual_income'] = 25000 + (df['feature_1'] - df['feature_1'].min()) / \
                         (df['feature_1'].max() - df['feature_1'].min()) * 225000
    
    # Create age between 21 and 70
    df['age'] = np.round(21 + (df['feature_2'] - df['feature_2'].min()) / \
                (df['feature_2'].max() - df['feature_2'].min()) * 49)
    
    # Employment length (0-30 years)
    df['employment_length'] = np.round(0 + (df['feature_3'] - df['feature_3'].min()) / \
                             (df['feature_3'].max() - df['feature_3'].min()) * 30)
    
    # Debt to income ratio (0.1 to 0.8)
    df['debt_to_income'] = 0.1 + (df['feature_4'] - df['feature_4'].min()) / \
                          (df['feature_4'].max() - df['feature_4'].min()) * 0.7
    
    # Credit used and credit limit
    df['credit_limit'] = 5000 + (df['feature_5'] - df['feature_5'].min()) / \
                        (df['feature_5'].max() - df['feature_5'].min()) * 45000
    
    df['credit_used'] = df['credit_limit'] * (0.1 + (df['feature_6'] - df['feature_6'].min()) / \
                       (df['feature_6'].max() - df['feature_6'].min()) * 0.9)
    
    # Monthly debt
    df['monthly_debt'] = df['annual_income'] * df['debt_to_income'] / 12
    
    # Recent inquiries (0-10)
    df['recent_inquiries'] = np.round(0 + (df['feature_7'] - df['feature_7'].min()) / \
                             (df['feature_7'].max() - df['feature_7'].min()) * 10)
    
    # Credit history length (1-30 years)
    df['credit_history_length'] = np.round(1 + (df['feature_8'] - df['feature_8'].min()) / \
                                 (df['feature_8'].max() - df['feature_8'].min()) * 29)
    
    # Monthly payment (derived from loan amount)
    # Assume 5 year loan with 10% interest rate
    rate = 0.1 / 12  # Monthly interest rate
    nper = 5 * 12    # Number of months (5 years)
    df['monthly_payment'] = df['loan_amount'] * (rate * (1 + rate)**nper) / ((1 + rate)**nper - 1)
    
    # Generate binary features
    df['has_mortgage'] = np.random.choice(['Yes', 'No'], size=n_samples, p=[0.4, 0.6])
    df['has_dependents'] = np.random.choice(['Yes', 'No'], size=n_samples, p=[0.3, 0.7])
    
    # Generate ZIP codes (random 5-digit numbers)
    df['zipcode'] = [f"{random.randint(10000, 99999)}" for _ in range(n_samples)]
    
    # Loan purpose categories
    loan_purposes = ['home_improvement', 'home_buying', 'debt_consolidation', 
                    'credit_card_refinancing', 'major_purchase', 'small_business',
                    'medical_expenses', 'vacation', 'moving', 'wedding', 
                    'car_financing', 'other']
    df['loan_purpose'] = [random.choice(loan_purposes) for _ in range(n_samples)]
    
    # Remove the raw features as they're not needed anymore
    df = df.drop(columns=[f'feature_{i}' for i in range(10)])
    
    # Save to CSV
    df.to_csv(file_path, index=False)
    print(f"Synthetic data saved to {file_path}")
    
    return df

# Check if credit_risk_data.csv already exists, if not generate it
if not os.path.exists('credit_risk_data.csv'):
    credit_data = generate_credit_risk_data(n_samples=10000)
else:
    print("Credit risk data file already exists. Loading existing data.")
    credit_data = pd.read_csv('credit_risk_data.csv')

# Display sample of the data
print("\nSample of credit risk data:")
print(credit_data.head())
print(f"\nShape: {credit_data.shape}")
print("\nClass distribution:")
print(credit_data['loan_default'].value_counts())
print(credit_data['loan_default'].value_counts(normalize=True).map("{:.2%}".format))
```

## Option 3: Use German Credit Data with Processing

If you prefer a smaller, established dataset, you can use the German Credit dataset:

```python
# Load and process German Credit Dataset
import pandas as pd
import numpy as np
import requests
from io import StringIO
import os

def download_german_credit_data(file_path='credit_risk_data.csv'):
    """
    Download and preprocess the German Credit dataset
    
    Parameters:
    -----------
    file_path: str, path to save the processed CSV file
    
    Returns:
    --------
    DataFrame with preprocessed credit risk data
    """
    print("Downloading and processing German Credit dataset...")
    
    # URL of the German Credit dataset
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
    
    try:
        # Download the file
        response = requests.get(url)
        response.raise_for_status()
        
        # Column names for the German Credit dataset
        columns = [
            'status', 'duration', 'credit_history', 'purpose', 'credit_amount',
            'savings', 'employment', 'installment_rate', 'personal_status_sex',
            'other_debtors', 'present_residence', 'property', 'age',
            'other_installment_plans', 'housing', 'existing_credits',
            'job', 'num_dependents', 'telephone', 'foreign_worker', 'class'
        ]
        
        # Parse the data (space-separated values)
        data = StringIO(response.text)
        df = pd.read_csv(data, sep=' ', header=None, names=columns)
        
        # Preprocess to match our expected format
        credit_data = pd.DataFrame()
        
        # Create Customer ID
        credit_data['Customer ID'] = ['CUST-' + str(i).zfill(4) for i in range(len(df))]
        
        # Target variable: original is 1=good, 2=bad, we convert to 0=good, 1=bad
        credit_data['loan_default'] = (df['class'] == 2).astype(int)
        
        # Create features corresponding to our expected schema
        credit_data['loan_amount'] = df['credit_amount']
        credit_data['annual_income'] = df['credit_amount'] * 2  # Approximation
        credit_data['age'] = df['age']
        
        # Map employment to employment_length
        employment_map = {'A71': 0, 'A72': 1, 'A73': 2, 'A74': 5, 'A75': 10}
        credit_data['employment_length'] = df['employment'].map(employment_map)
        
        # Create debt to income
        credit_data['debt_to_income'] = df['installment_rate'] / 100
        
        # Create credit_used and credit_limit
        credit_data['credit_limit'] = df['credit_amount'] * 1.5  # Approximation
        credit_data['credit_used'] = df['credit_amount']
        
        # Create monthly debt
        credit_data['monthly_debt'] = df['credit_amount'] / df['duration']
        
        # Create other fields
        credit_data['recent_inquiries'] = df['existing_credits']
        credit_data['credit_history_length'] = np.random.randint(1, 20, size=len(df))
        credit_data['monthly_payment'] = df['credit_amount'] / df['duration']
        credit_data['zipcode'] = np.random.randint(10000, 99999, size=len(df)).astype(str)
        
        # Binary fields
        credit_data['has_mortgage'] = np.where(df['property'] == 'A121', 'Yes', 'No')
        credit_data['has_dependents'] = np.where(df['num_dependents'] > 0, 'Yes', 'No')
        
        # Loan purpose
        purpose_map = {
            'A40': 'car_financing', 'A41': 'appliances', 'A42': 'education',
            'A43': 'vacation', 'A44': 'professional_training', 'A45': 'small_business',
            'A46': 'medical_expenses', 'A47': 'home_improvement', 'A48': 'other',
            'A49': 'car_financing', 'A410': 'other'
        }
        credit_data['loan_purpose'] = df['purpose'].map(purpose_map)
        
        # Save the processed data
        credit_data.to_csv(file_path, index=False)
        print(f"German Credit data processed and saved to {file_path}")
        
        return credit_data
        
    except Exception as e:
        print(f"Error downloading or processing the German Credit dataset: {e}")
        print("Falling back to synthetic data generation...")
        return generate_credit_risk_data(1000, file_path)

# Check if credit_risk_data.csv already exists, if not download and process it
if not os.path.exists('credit_risk_data.csv'):
    credit_data = download_german_credit_data()
else:
    print("Credit risk data file already exists. Loading existing data.")
    credit_data = pd.read_csv('credit_risk_data.csv')

# Display sample of the data
print("\nSample of credit risk data:")
print(credit_data.head())
print(f"\nShape: {credit_data.shape}")
print("\nClass distribution:")
print(credit_data['loan_default'].value_counts())
print(credit_data['loan_default'].value_counts(normalize=True).map("{:.2%}".format))
```

## Using the Data with the Classification Code

After obtaining or generating the data using one of the methods above, you can run the classification code from the case study. The data should now be properly formatted with all the required columns.

### Summary of Options

1. **For larger datasets with more features**: Use the Lending Club dataset from Kaggle
2. **For medium-sized datasets**: Use the Home Credit Default Risk dataset
3. **For smaller datasets**: Use the processed German Credit dataset
4. **If you can't download external data**: Use the synthetic data generator

The synthetic data generator is the most convenient option as it:
- Creates all required fields with realistic values
- Ensures proper distributions for numerical and categorical features
- Includes class imbalance (15% default rate) to reflect real-world credit risk scenarios
- Doesn't require downloading large files from external sources

Simply place the data generation code at the beginning of your script before loading the data with `pd.read_csv('credit_risk_data.csv')`.

# CREATE 'credit_risk_data.csv'

In [4]:
# !pip install Faker

In [5]:
# Generate synthetic credit risk data
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from faker import Faker
import random
import os

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)
fake = Faker()
Faker.seed(42)

def generate_credit_risk_data(n_samples=10000, file_path='credit_risk_data.csv'):
    """
    Generate synthetic credit risk data and save as CSV
    
    Parameters:
    -----------
    n_samples: int, number of samples to generate
    file_path: str, path to save the CSV file
    
    Returns:
    --------
    DataFrame with synthetic credit risk data
    """
    print(f"Generating synthetic credit risk dataset with {n_samples} samples...")
    
    # Generate binary target with class imbalance (15% default rate)
    X, y = make_classification(
        n_samples=n_samples,
        n_features=10,
        n_informative=8,
        n_redundant=2,
        weights=[0.85, 0.15],  # 15% default rate
        random_state=42
    )
    
    # Create dataframe with features
    df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(10)])
    
    # Add customer IDs
    df['Customer ID'] = [f'CUST-{i:05d}' for i in range(n_samples)]
    
    # Add loan_default target column
    df['loan_default'] = y
    
    # Generate realistic features based on the synthetic data
    # Scale feature_0 to loan_amount (5,000 to 50,000)
    df['loan_amount'] = 5000 + (df['feature_0'] - df['feature_0'].min()) / \
                       (df['feature_0'].max() - df['feature_0'].min()) * 45000
    
    # Scale feature_1 to annual_income (25,000 to 250,000)
    df['annual_income'] = 25000 + (df['feature_1'] - df['feature_1'].min()) / \
                         (df['feature_1'].max() - df['feature_1'].min()) * 225000
    
    # Create age between 21 and 70
    df['age'] = np.round(21 + (df['feature_2'] - df['feature_2'].min()) / \
                (df['feature_2'].max() - df['feature_2'].min()) * 49)
    
    # Employment length (0-30 years)
    df['employment_length'] = np.round(0 + (df['feature_3'] - df['feature_3'].min()) / \
                             (df['feature_3'].max() - df['feature_3'].min()) * 30)
    
    # Debt to income ratio (0.1 to 0.8)
    df['debt_to_income'] = 0.1 + (df['feature_4'] - df['feature_4'].min()) / \
                          (df['feature_4'].max() - df['feature_4'].min()) * 0.7
    
    # Credit used and credit limit
    df['credit_limit'] = 5000 + (df['feature_5'] - df['feature_5'].min()) / \
                        (df['feature_5'].max() - df['feature_5'].min()) * 45000
    
    df['credit_used'] = df['credit_limit'] * (0.1 + (df['feature_6'] - df['feature_6'].min()) / \
                       (df['feature_6'].max() - df['feature_6'].min()) * 0.9)
    
    # Monthly debt
    df['monthly_debt'] = df['annual_income'] * df['debt_to_income'] / 12
    
    # Recent inquiries (0-10)
    df['recent_inquiries'] = np.round(0 + (df['feature_7'] - df['feature_7'].min()) / \
                             (df['feature_7'].max() - df['feature_7'].min()) * 10)
    
    # Credit history length (1-30 years)
    df['credit_history_length'] = np.round(1 + (df['feature_8'] - df['feature_8'].min()) / \
                                 (df['feature_8'].max() - df['feature_8'].min()) * 29)
    
    # Monthly payment (derived from loan amount)
    # Assume 5 year loan with 10% interest rate
    rate = 0.1 / 12  # Monthly interest rate
    nper = 5 * 12    # Number of months (5 years)
    df['monthly_payment'] = df['loan_amount'] * (rate * (1 + rate)**nper) / ((1 + rate)**nper - 1)
    
    # Generate binary features
    df['has_mortgage'] = np.random.choice(['Yes', 'No'], size=n_samples, p=[0.4, 0.6])
    df['has_dependents'] = np.random.choice(['Yes', 'No'], size=n_samples, p=[0.3, 0.7])
    
    # Generate ZIP codes (random 5-digit numbers)
    df['zipcode'] = [f"{random.randint(10000, 99999)}" for _ in range(n_samples)]
    
    # Loan purpose categories
    loan_purposes = ['home_improvement', 'home_buying', 'debt_consolidation', 
                    'credit_card_refinancing', 'major_purchase', 'small_business',
                    'medical_expenses', 'vacation', 'moving', 'wedding', 
                    'car_financing', 'other']
    df['loan_purpose'] = [random.choice(loan_purposes) for _ in range(n_samples)]
    
    # Remove the raw features as they're not needed anymore
    df = df.drop(columns=[f'feature_{i}' for i in range(10)])
    
    # Save to CSV
    df.to_csv(file_path, index=False)
    print(f"Synthetic data saved to {file_path}")
    
    return df

# Check if credit_risk_data.csv already exists, if not generate it
if not os.path.exists('credit_risk_data.csv'):
    credit_data = generate_credit_risk_data(n_samples=10000)
else:
    print("Credit risk data file already exists. Loading existing data.")
    credit_data = pd.read_csv('credit_risk_data.csv')

# Display sample of the data
print("\nSample of credit risk data:")
print(credit_data.head())
print(f"\nShape: {credit_data.shape}")
print("\nClass distribution:")
print(credit_data['loan_default'].value_counts())
print(credit_data['loan_default'].value_counts(normalize=True).map("{:.2%}".format))

Generating synthetic credit risk dataset with 10000 samples...
Synthetic data saved to credit_risk_data.csv

Sample of credit risk data:
  Customer ID  loan_default   loan_amount  annual_income   age  \
0  CUST-00000             0  23430.189415  116301.165754  45.0   
1  CUST-00001             0  24533.850886   92876.714947  49.0   
2  CUST-00002             0  32162.586016  154009.942350  49.0   
3  CUST-00003             0  27711.722187   74817.696802  50.0   
4  CUST-00004             0  24803.000930  138171.462423  48.0   

   employment_length  debt_to_income  credit_limit   credit_used  \
0               12.0        0.475836  31962.566391  21733.793191   
1               10.0        0.341414  24450.543625  17570.588636   
2               20.0        0.354333  28401.223552  21950.778491   
3               15.0        0.471411  15348.141710  10539.878566   
4               18.0        0.382820  35349.620548  27884.072514   

   monthly_debt  recent_inquiries  credit_history_length 